<a href="https://colab.research.google.com/github/chandabi/bhoomika/blob/main/handdigit22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio tensorflow-datasets importlib-resources
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import gradio as gr
from PIL import Image, ImageOps
from tensorflow.keras import layers, models
(x_tr_d, y_tr_d), (x_te_d, y_te_d) = tf.keras.datasets.mnist.load_data()
def load_emnist_letters(split):
    ds = tfds.load("emnist/letters", split=split, batch_size=-1, as_supervised=True)
    imgs, labels = tfds.as_numpy(ds)
    imgs = imgs.squeeze(-1)
    imgs = np.transpose(imgs, (0, 2, 1))
    labels = labels.astype(np.int64)
    keep = labels >= 1
    return imgs[keep], labels[keep] - 1 + 10
x_y_tr_l = load_emnist_letters("train")
x_te_l, y_te_l = load_emnist_letters("test")
x_train = np.concatenate([x_tr_d, x_tr_l]).astype("float32") / 255.0
y_train = np.concatenate([y_tr_d, y_tr_l])
x_test = np.concatenate([x_te_d, x_te_l]).astype("float32") / 255.0
y_test = np.concatenate([y_te_d, y_te_l])
x_train = x_train[..., None]
x_test = x_test[..., None]
idx = np.random.permutation(len(x_train))
x_train, y_train = x_train[idx], y_train[idx]
print("Train:", x_train.shape, "Test:", x_test.shape, "Classes:", len(np.unique(y_train)))
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(36, activation="softmax"),
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1,
)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nTest accuracy: {test_acc * 100:.2f}%")
model.save("/content/handwriting_model.keras")
print("Model saved to /content/handwriting_model.keras")
def class_to_char(c):
    return str(c) if c < 10 else chr(ord("A") + c - 10)
def to_pil(value):
    if value is None:
        return None
    if isinstance(value, dict):
        value = value.get("composite")
        if value is None:
            return None
    img = Image.fromarray(value) if isinstance(value, np.ndarray) else value
    if img.mode in ("RGBA", "LA"):
        bg = Image.new("RGBA", img.size, (255, 255, 255, 255))
        img = Image.alpha_composite(bg, img.convert("RGBA"))
    return img.convert("L")
def predict(value):
    img = to_pil(value)
    if img is None:
        return None, {}, "Draw or upload something first."
    img = ImageOps.invert(img)
    bbox = img.getbbox()
    if bbox is None:
        return None, {}, "The image looks blank."
    img = img.crop(bbox)
    img.thumbnail((20, 20))
    canvas = Image.new("L", (28, 28), 0)
    canvas.paste(img, ((28 - img.width) // 2, (28 - img.height) // 2))

    arr = np.array(canvas, dtype=np.float32).reshape(1, 28, 28, 1) / 255.0
    probs = model.predict(arr, verbose=0)[0]
    top5 = np.argsort(probs)[::-1][:5]
    scores = {class_to_char(int(c)): float(probs[c]) for c in top5}

    best = int(top5[0])
    kind = "Digit" if best < 10 else "Alphabet"
    text = (
        f"### Prediction: **{class_to_char(best)}** ({kind}, {probs[best] * 100:.1f}%)\n"
        f"Digit vs alphabet: {probs[:10].sum() * 100:.1f}% digit | {probs[10:].sum() * 100:.1f}% alphabet\n\n"
        f"Best digit: **{int(np.argmax(probs[:10]))}** ({probs[:10].max() * 100:.1f}%)  \n"
        f"Best alphabet: **{chr(ord('A') + int(np.argmax(probs[10:])))}** ({probs[10:].max() * 100:.1f}%)"
    )
    return canvas.resize((196, 196), Image.NEAREST), scores, text
with gr.Blocks(title="Handwritten Digit & Alphabet Recognition") as demo:
    gr.Markdown("# ✍️ Handwritten Digit & Alphabet Recognition\nDraw ONE digit or capital letter, or upload an image.")
    with gr.Row():
        with gr.Column():
            with gr.Tab("Draw"):
                sketch = gr.Sketchpad(type="pil", image_mode="RGBA", canvas_size=(320, 320))
                b1 = gr.Button("Recognize", variant="primary")
            with gr.Tab("Upload"):
                upload = gr.Image(type="pil", label="Dark ink on light paper")
                b2 = gr.Button("Recognize", variant="primary")
        with gr.Column():
            summary = gr.Markdown()
            top5_label = gr.Label(num_top_classes=5, label="Top 5 predictions")
            seen = gr.Image(label="What the model sees (28x28)", interactive=False)
    outs = [seen, top5_label, summary]
    b1.click(predict, sketch, outs)
    b2.click(predict, upload, outs)

demo.launch(share=True, debug=True)

Train: (148800, 28, 28, 1) Test: (24800, 28, 28, 1) Classes: 36
Epoch 1/5
1047/1047 ━━━━━━━━━━━━━━━━━━━━ 109s 102ms/step - accuracy: 0.7994 - loss: 0.6737 - val_accuracy: 0.9198 - val_loss: 0.2528
Epoch 2/5
1047/1047 ━━━━━━━━━━━━━━━━━━━━ 106s 101ms/step - accuracy: 0.9075 - loss: 0.2946 - val_accuracy: 0.9370 - val_loss: 0.1981
Epoch 3/5
1047/1047 ━━━━━━━━━━━━━━━━━━━━ 104s 100ms/step - accuracy: 0.9249 - loss: 0.2354 - val_accuracy: 0.9429 - val_loss: 0.1752
Epoch 4/5
1047/1047 ━━━━━━━━━━━━━━━━━━━━ 106s 101ms/step - accuracy: 0.9350 - loss: 0.2001 - val_accuracy: 0.9476 - val_loss: 0.1642
Epoch 5/5
1047/1047 ━━━━━━━━━━━━━━━━━━━━ 112s 107ms/step - accuracy: 0.9418 - loss: 0.1770 - val_accuracy: 0.9520 - val_loss: 0.1508

Test accuracy: 94.49%
Model saved to /content/handwriting_model.keras
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://15311b1a6a0e21a17e.gradio.live


In [ ]:
!pip install streamlit

import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image

st.set_page_config(page_title="Skin Image Classifier")

st.title("🩺 Skin Disease Image Classifier")
st.write("Upload a skin image for an experimental AI classification.")

@st.cache_resource
def load_model():
    return tf.keras.models.load_model("skin_model.keras")

try:
    model = load_model()

    uploaded_file = st.file_uploader(
        "Upload a skin image",
        type=["jpg", "jpeg", "png"]
    )

    if uploaded_file is not None:
        image = Image.open(uploaded_file).convert("RGB")
        st.image(image, caption="Uploaded Image", width=300)

        image = image.resize((180, 180))
        image_array = np.array(image, dtype=np.float32)
        image_array = np.expand_dims(image_array, axis=0)

        if st.button("Classify Image"):
            prediction = model.predict(image_array, verbose=0)[0]

            predicted_class = int(np.argmax(prediction))
            confidence = float(prediction[predicted_class]) * 100

            st.subheader("Model Prediction")
            st.write(f"Category: {predicted_class + 1}")
            st.write(f"Model confidence: {confidence:.2f}%")

            st.warning(
                "This is an experimental prediction, not a medical "
                "diagnosis. Please consult a qualified dermatologist."
            )

except Exception as error:
    st.error(
        "Model not found or could not be loaded. "
        "Run train_model.py first.")